# P129 — MusicLM: generar música a partir de texto

## 1. Título y paper

**Paper:** *MusicLM: Generating Music From Text*  
**Autoría:** Andrea Agostinelli, Timo I. Denk, Zalán Borsos, Jesse Engel, Mauro Verzetti, Antoine Caillon, y otros  
**Año y venue:** 2023 · arXiv:2301.11325  
**Nivel:** L3 · **Motor:** `musiclm`  
**Ficha completa:** [`P129_musiclm`](../../papers/foundational/P129_musiclm/README.md)

**Hito:** Genera música coherente de varios minutos desde una descripción en lenguaje natural, y publica MusicCaps para que la tarea se pueda evaluar.

- [arXiv:2301.11325](https://arxiv.org/abs/2301.11325)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los pares texto-música son escasísimos comparados con los pares texto-imagen, y la música tiene estructura a escalas que no caben en una sola ventana de contexto: el timbre se juega en milisegundos y la forma, en minutos.
2. Ejecutar una implementación mínima de la propuesta: Una jerarquía de dos tipos de token —semánticos, a baja frecuencia, que llevan la estructura, y acústicos, a alta frecuencia, que llevan el detalle— y un entrenamiento que aprovecha audio sin etiquetar mediante una representación conjunta de texto y música.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P127
- P18


## 4. Intuición

Una pieza vuelve a su motivo inicial en el compás 64. Con tokens acústicos, la ventana del modelo llega a 20 compases: cuando toca reexponer, no queda ni rastro de qué había que reexponer.


## 5. Concepto mínimo

```text
acústico  : 50 tokens/compás  → ventana 1024 abarca  20,5 compases
semántico :  2 tokens/compás  → ventana 1024 abarca 512   compases

No son dos calidades. Son dos HORIZONTES.
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('musiclm', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Cuántos compases abarca cada escala?
2. ¿Ve el modelo acústico la exposición original al reexponer?
3. ¿Cuántos pares texto-música hay para entrenar?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('musiclm', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('musiclm', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Acústico: **20,5** compases. Semántico: **512**. Durante la reexposición, la escala acústica alcanza la exposición original en **0 de 32** compases y la semántica en **32 de 32**. Y los datos: incluso con 4 descripciones por clip, el conjunto anotado son **22 000** pares.


## 10. Comentario pedagógico

Esa escasez es la mitad del problema y casi nunca se menciona. Frente a los miles de millones de pares texto-imagen que hay en la web, describir música con palabras es un recurso raro — porque casi nadie escribe al lado de una canción qué instrumentos suenan y en qué tempo. De ahí que el artículo publique MusicCaps además del modelo.


## 11. Error o anti-patrón deliberado

Anti-patrón: creer que basta con una ventana de contexto mayor.


In [ ]:
print('Doblar la ventana acustica pasa de 20 a 41 compases: sigue sin llegar a 64.')
print('La escala semantica llega con la MISMA ventana, porque gasta 25 veces menos tokens.')
print('Es un problema de representacion, no de tamano.')

## 12. Corrección

Los dos horizontes, medidos:


In [ ]:
r = run_paper_lab('musiclm', seed=3)['result']
print('forma:', r['pieza'])
print('alcance acustico:', r['alcance_acustico_en_compases'])
print('alcance semantico:', r['alcance_semantico_en_compases'])
print('solo acustico:', r['coherencia_solo_acustico'])
print('con semantico:', r['coherencia_con_semantico'])

## 13. Desafío guiado

Explica por qué una jerarquía de dos escalas resuelve el problema mejor que agrandar la ventana, y qué coste tiene.


In [ ]:
r = run_paper_lab('musiclm', seed=3)['result']
show(r)

## 14. Desafío autónomo

Elige una pieza con forma clara —ABA, rondó, tema y variaciones— y mide en segundos la distancia entre la exposición y la reexposición. Compárala con la ventana de un modelo que uses.


## 15. Evidencia de aprendizaje

Guarda la medición y si la ventana llegaría o no.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P129_musiclm/README.md) · evaluación formal: [`assessments/papers/P129_musiclm.md`](../../assessments/papers/P129_musiclm.md)


## 16. Cierre

La música ya se genera. Lo siguiente es la voz, donde copiar a alguien deja de ser un problema técnico y pasa a ser uno de identidad.


## 17. Conexión con el siguiente hito



Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
